# Week 8 Presentation Brief — Variation B
## 🦘 Dingo-Kangaroo Dynamics: Predator-Prey Modelling
**SCIE1500 — Semester 2, 2026 | Group 1 (second presentation) — presenting in Week 9**

> Work through all parts during the Week 8 lab. Your **10-minute Week 9 presentation** should cover: the ODE model, equilibria, phase plane, and management implications for dingo control
> **What to submit:** upload your completed presentation slides (PDF or PowerPoint) to the LMS after your presentation.


---
## 📋 Scenario

In the Kimberley, dingoes prey on kangaroos. Population dynamics:

$$\frac{dK}{dt} = 0.5K - 0.0015KD$$

$$\frac{dD}{dt} = -0.25D + 0.0008KD$$

where $K$ = kangaroo population and $D$ = dingo population.

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Derive and interpret the coexistence equilibrium | ~15 min |
| B | Plot the phase plane and simulate trajectories using odeint | ~30 min |
| C | Simulate a dingo culling scenario and evaluate long-run outcomes | ~15 min |

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
from scipy.integrate import odeint

r_K = 0.5;  alpha = 0.0015   # kangaroo growth, predation
d_D = 0.25; beta  = 0.0008   # dingo death, conversion efficiency

# Coexistence equilibrium
K_star = d_D / beta    # = 0.25/0.0008 = 312.5
D_star = r_K / alpha   # = 0.5/0.0015 ≈ 333.3

print(f"Coexistence equilibrium:")
print(f"  K* = {K_star:.1f} kangaroos")
print(f"  D* = {D_star:.1f} dingoes")

def dingo_kangaroo(state, t):
    K, D = state
    dK = r_K*K  - alpha*K*D
    dD = -d_D*D + beta*K*D
    return [dK, dD]

Phase plane with odeint trajectories.

In [ ]:
# Phase plane with odeint trajectories
t_span = np.linspace(0, 50, 3000)
ICs = [(500, 100), (300, 400), (600, 300)]
colors = ["steelblue", "firebrick", "seagreen"]
labels = ["K₀=500,D₀=100", "K₀=300,D₀=400", "K₀=600,D₀=300"]

K_grid = np.linspace(0, 900, 18)
D_grid = np.linspace(0, 600, 18)
KK, DD = np.meshgrid(K_grid, D_grid)
dK_grid = r_K*KK - alpha*KK*DD
dD_grid = -d_D*DD + beta*KK*DD
mag = np.sqrt(dK_grid**2 + dD_grid**2) + 1e-6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.quiver(KK, DD, dK_grid/mag, dD_grid/mag, mag, cmap="Greys", alpha=0.5)
for ic, col, lbl in zip(ICs, colors, labels):
    sol = odeint(dingo_kangaroo, ic, t_span)
    ax1.plot(sol[:, 0], sol[:, 1], col, lw=2, label=lbl)
    ax1.plot(ic[0], ic[1], "o", color=col, ms=8)
ax1.axvline(K_star, color="k", ls="--", alpha=0.5, label=f"K*={K_star:.0f}")
ax1.axhline(D_star, color="k", ls=":",  alpha=0.5, label=f"D*={D_star:.0f}")
ax1.plot(K_star, D_star, "k*", ms=14, zorder=5, label="Equilibrium")
ax1.set_xlabel("Kangaroos K"); ax1.set_ylabel("Dingoes D")
ax1.set_title("Dingo–Kangaroo Phase Plane"); ax1.legend(fontsize=7)

sol0 = odeint(dingo_kangaroo, ICs[0], t_span)
ax2.plot(t_span, sol0[:, 0], "steelblue", lw=2, label="Kangaroos")
ax2.plot(t_span, sol0[:, 1], "firebrick", lw=2, label="Dingoes")
ax2.axhline(K_star, color="steelblue", ls="--", alpha=0.4)
ax2.axhline(D_star, color="firebrick", ls="--", alpha=0.4)
ax2.set_xlabel("Years"); ax2.set_ylabel("Population")
ax2.set_title("Time Series (K₀=500, D₀=100)"); ax2.legend()

plt.tight_layout(); plt.show()

Dingo culling scenario: remove 30 dingoes/year.

In [ ]:
# Dingo culling scenario: remove 30 dingoes/year
cull_rate = 30

N_steps = 3000
dt = 50 / N_steps
K_c = np.zeros(N_steps + 1); D_c = np.zeros(N_steps + 1); t_c = np.zeros(N_steps + 1)
K_c[0], D_c[0] = 400, 200

for i in range(N_steps):
    dK = r_K*K_c[i] - alpha*K_c[i]*D_c[i]
    dD = -d_D*D_c[i] + beta*K_c[i]*D_c[i] - cull_rate
    K_c[i+1] = max(0, K_c[i] + dK * dt)
    D_c[i+1] = max(0, D_c[i] + dD * dt)
    t_c[i+1] = t_c[i] + dt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_c, K_c, "steelblue", lw=2, label="Kangaroos (culled)")
ax.plot(t_c, D_c, "firebrick", lw=2, label="Dingoes (culled)")
ax.axhline(K_star, color="steelblue", ls="--", alpha=0.4)
ax.axhline(D_star, color="firebrick", ls="--", alpha=0.4)
ax.set_xlabel("Years"); ax.set_ylabel("Population")
ax.set_title(f"Dingo Culling: Remove {cull_rate} dingoes/yr")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Year 50: K = {K_c[-1]:.0f}, D = {D_c[-1]:.0f}")
print(f"Without culling eq: K* = {K_star:.0f}, D* = {D_star:.0f}")

✏️ **Management Discussion:**

1. Does culling 30 dingoes/year lead to dingo extinction or a new equilibrium? What does this imply for kangaroo population management?
2. What cull rate would be needed to reduce dingo population below 100?
3. What are the ecological risks of reducing dingo numbers in the Kimberley?

```
DISCUSSION:
...
```

---
## ✅ Presentation Checklist (Week 9, 10 minutes)

1. **ODEs** (~2 min): State and interpret each term in the dingo-kangaroo model.
2. **Equilibrium** (~2 min): Derive $K^*$, $D^*$; compare to feral cat model parameters.
3. **Phase plane** (~4 min): Show plots; trace one orbit; explain oscillation pattern.
4. **Culling** (~2 min): Is culling effective? What are the long-run implications?